## 擦除断层处解释点

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 设置中文字体
plt.rcParams["font.family"] = "SimHei"  # 黑体 SimHei 支持中文
plt.rcParams["axes.unicode_minus"] = False  # 正常显示负号

In [ ]:
# ==================== 层位配置 ====================
# 修改这里可以更换不同的层位
SURFACE_NAME = "H3-1"
# ================================================

In [ ]:
# 路径配置
data_dir = "../../data/interpretation"
interpre_dir = os.path.join(data_dir, "interpre")  # 解释数据文件夹
fault_dir = os.path.join(data_dir, "fault")  # 断层数据文件夹
output_dir = f"{SURFACE_NAME.replace('-', '_')}_batch_output"

# 创建输出目录
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"当前处理层位: {SURFACE_NAME}")
print(f"解释数据目录: {interpre_dir}")
print(f"断层数据目录: {fault_dir}")
print(f"输出目录: {output_dir}")

In [ ]:
def load_interpretation_data(filepath):
    """
    加载解释文件数据

    Args:
        filepath (str): 解释文件路径

    Returns:
        np.ndarray: shape为(n, 5)的数组，每行为[inline, xline, x, y, z]
    """
    data = []

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue

                try:
                    # 解析格式: INLINE :    235 XLINE :    984    682449.04042   3215396.36483      2261.35669
                    parts = line.split()
                    if (
                        len(parts) >= 8
                        and parts[0] == "INLINE"
                        and parts[1] == ":"
                        and parts[3] == "XLINE"
                        and parts[4] == ":"
                    ):
                        inline = int(parts[2])
                        xline = int(parts[5])
                        x = float(parts[6])
                        y = float(parts[7])
                        z = float(parts[8])
                        data.append([inline, xline, x, y, z])
                    else:
                        print(f"警告: 第{line_num}行格式不正确，跳过: {line}")
                        print(f"  解析结果: {parts}")  # 调试信息

                except (ValueError, IndexError) as e:
                    print(f"警告: 第{line_num}行数据解析错误，跳过: {line}, 错误: {e}")

    except FileNotFoundError:
        print(f"错误: 找不到文件 {filepath}")
        return np.array([])
    except Exception as e:
        print(f"错误: 读取文件时发生异常: {e}")
        return np.array([])

    if not data:
        print("警告: 未读取到有效数据")
        return np.array([])

    result = np.array(data)
    print(f"成功加载解释数据: {len(result)} 个点")
    return result


def remove_interpretation_outliers_by_z(interpretation_data, method="iqr", factor=1.5):
    """
    根据Z值去除离群值

    Args:
        interpretation_data (np.ndarray): 解释数据
        method (str): 离群值检测方法 ('iqr' 或 'zscore')
        factor (float): 离群值判定因子

    Returns:
        np.ndarray: 去除离群值后的数据
    """
    if len(interpretation_data) == 0:
        return interpretation_data

    z_values = interpretation_data[:, 4]
    original_count = len(interpretation_data)

    if method == "iqr":
        # 使用四分位距方法
        Q1 = np.percentile(z_values, 25)
        Q3 = np.percentile(z_values, 75)
        IQR = Q3 - Q1

        lower_bound = Q1 - factor * IQR
        upper_bound = Q3 + factor * IQR

        mask = (z_values >= lower_bound) & (z_values <= upper_bound)

        print(f"IQR离群值检测:")
        print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
        print(f"  有效Z值范围: [{lower_bound:.2f}, {upper_bound:.2f}]")

    elif method == "zscore":
        # 使用Z-score方法
        mean_z = np.mean(z_values)
        std_z = np.std(z_values)

        z_scores = np.abs((z_values - mean_z) / std_z)
        mask = z_scores < factor

        print(f"Z-score离群值检测:")
        print(f"  均值: {mean_z:.2f}, 标准差: {std_z:.2f}")
        print(f"  Z-score阈值: {factor}")

    else:
        print(f"错误: 不支持的离群值检测方法 '{method}'")
        return interpretation_data

    filtered_data = interpretation_data[mask]
    removed_count = original_count - len(filtered_data)

    print(f"  原始数据: {original_count} 个点")
    print(f"  移除离群值: {removed_count} 个点 ({removed_count / original_count * 100:.1f}%)")
    print(f"  保留数据: {len(filtered_data)} 个点")

    if len(filtered_data) > 0:
        print(f"  过滤后Z值范围: {filtered_data[:, 4].min():.2f} ~ {filtered_data[:, 4].max():.2f}")

    return filtered_data


def get_fault_files(fault_dir):
    """
    获取断层文件夹中的所有断层文件

    Args:
        fault_dir (str): 断层文件夹路径

    Returns:
        list: 断层文件路径列表
    """
    # 支持多种可能的文件扩展名
    patterns = [
        os.path.join(fault_dir, "*"),
        os.path.join(fault_dir, "*.txt"),
        os.path.join(fault_dir, "*.dat"),
        os.path.join(fault_dir, "*surface*"),
        os.path.join(fault_dir, "*fault*"),
        os.path.join(fault_dir, "F*"),
    ]

    fault_files = []
    for pattern in patterns:
        files = glob.glob(pattern)
        for file in files:
            if os.path.isfile(file) and file not in fault_files:
                fault_files.append(file)

    # 按文件名排序
    fault_files.sort()

    print(f"发现 {len(fault_files)} 个断层文件:")
    for i, file in enumerate(fault_files, 1):
        filename = os.path.basename(file)
        print(f"  {i}. {filename}")

    return fault_files


def load_fault_points(filepath):
    """
    加载简单格式的断层点数据

    Args:
        filepath (str): 断层文件路径

    Returns:
        tuple: (fault_x, fault_y, fault_z) 断层点坐标数组
    """
    try:
        fault_points = []

        with open(filepath, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue

                try:
                    # 解析格式: x y z
                    parts = line.split()
                    if len(parts) >= 3:
                        x = float(parts[0])
                        y = float(parts[1])
                        z = float(parts[2])
                        fault_points.append([x, y, z])
                    else:
                        print(f"警告: 第{line_num}行格式不正确，跳过: {line}")

                except (ValueError, IndexError) as e:
                    print(f"警告: 第{line_num}行数据解析错误，跳过: {line}, 错误: {e}")

        if not fault_points:
            print("警告: 未读取到有效的断层点数据")
            return np.array([]), np.array([]), np.array([])

        fault_points = np.array(fault_points)
        fault_x = fault_points[:, 0]
        fault_y = fault_points[:, 1]
        fault_z = fault_points[:, 2]

        print(f"成功加载断层点数据:")
        print(f"  点数量: {len(fault_points)}")
        print(f"  X范围: {fault_x.min():.2f} ~ {fault_x.max():.2f}")
        print(f"  Y范围: {fault_y.min():.2f} ~ {fault_y.max():.2f}")
        print(f"  Z范围: {fault_z.min():.2f} ~ {fault_z.max():.2f}")

        return fault_x, fault_y, fault_z

    except FileNotFoundError:
        print(f"错误: 找不到文件 {filepath}")
        return np.array([]), np.array([]), np.array([])
    except Exception as e:
        print(f"错误: 读取断层文件时发生异常: {e}")
        return np.array([]), np.array([]), np.array([])


def filter_fault_points_by_z(fault_x, fault_y, fault_z, z_min, z_max, z_buffer_ratio=0.1):
    """
    根据Z值范围过滤断层点数据

    Args:
        fault_x, fault_y, fault_z: 断层点坐标
        z_min, z_max: 解释数据的Z值范围
        z_buffer_ratio: Z方向缓冲区比例

    Returns:
        tuple: 过滤后的断层点坐标 (filtered_fault_x, filtered_fault_y, filtered_fault_z)
    """
    if len(fault_x) == 0:
        return np.array([]), np.array([]), np.array([])

    # 计算Z值缓冲区
    z_range = z_max - z_min
    z_buffer = z_range * z_buffer_ratio
    extended_z_min = z_min - z_buffer
    extended_z_max = z_max + z_buffer

    # 找到Z值在有效范围内的点
    valid_mask = (fault_z >= extended_z_min) & (fault_z <= extended_z_max)

    # 提取有效点的坐标
    filtered_fault_x = fault_x[valid_mask]
    filtered_fault_y = fault_y[valid_mask]
    filtered_fault_z = fault_z[valid_mask]

    print(f"断层点Z值过滤:")
    print(f"  过滤范围: Z ∈ [{extended_z_min:.2f}, {extended_z_max:.2f}] (含{z_buffer_ratio * 100}%缓冲)")
    print(f"  原始断层点数: {len(fault_x)}")
    print(f"  有效断层点数: {len(filtered_fault_x)}")

    return filtered_fault_x, filtered_fault_y, filtered_fault_z


In [ ]:
# def load_surface_data_top_left(filepath):
#     """
#     加载Petrel Surface文件数据 - 专用于从左上角开始的坐标排列

#     Args:
#         filepath (str): Surface文件路径

#     Returns:
#         tuple: (x_coords, y_coords, z_grid) 其中z_grid是2D数组
#     """
#     try:
#         with open(filepath, "r", encoding="utf-8") as f:
#             lines = f.readlines()

#         # 解析头部信息
#         fslimi_line = None
#         fsnrow_line = None
#         fsxinc_line = None
#         z_data_start = None

#         for i, line in enumerate(lines):
#             line = line.strip()
#             if line.startswith("FSLIMI"):
#                 fslimi_line = line
#             elif line.startswith("FSNROW"):
#                 fsnrow_line = line
#             elif line.startswith("FSXINC"):
#                 fsxinc_line = line
#             elif line.startswith("->MSMODL:"):
#                 z_data_start = i + 1
#                 break

#         if not all([fslimi_line, fsnrow_line, fsxinc_line, z_data_start is not None]):
#             raise ValueError("Surface文件格式不完整，缺少必要的头部信息")

#         # 解析边界信息 FSLIMI
#         fslimi_parts = fslimi_line.split()[1:]  # 跳过'FSLIMI'
#         x_min, x_max, y_min, y_max, z_min, z_max = map(float, fslimi_parts)

#         # 解析网格尺寸 FSNROW
#         fsnrow_parts = fsnrow_line.split()[1:]  # 跳过'FSNROW'
#         nx, ny = map(int, fsnrow_parts)  # nx是X方向(列数), ny是Y方向(行数)

#         # 解析增量 FSXINC
#         fsxinc_parts = fsxinc_line.split()[1:]  # 跳过'FSXINC'
#         dx, dy = map(float, fsxinc_parts)

#         # 生成坐标网格 - 修复：直接使用头部信息的边界
#         x_coords = np.linspace(x_min, x_max, nx)  # 修复：使用x_max而不是计算
#         y_coords = np.linspace(y_max, y_min, ny)  # 修复：从y_max到y_min

#         # 读取Z值数据
#         z_data_lines = lines[z_data_start:]
#         z_values = []

#         for line in z_data_lines:
#             line = line.strip()
#             if line:
#                 values = line.split()
#                 z_values.extend([float(v) for v in values])

#         expected_size = nx * ny
#         if len(z_values) != expected_size:
#             print(f"警告: Z值数量({len(z_values)})与预期网格大小({expected_size})不匹配")

#         # 将Z值重塑为2D网格 (行优先排列)
#         z_grid = np.array(z_values[:expected_size]).reshape(ny, nx)

#         print(f"成功加载Surface数据:")
#         print(f"  网格尺寸: {nx} × {ny}")
#         print(f"  X范围: {x_min:.2f} ~ {x_max:.2f}")
#         print(f"  Y范围: {y_min:.2f} ~ {y_max:.2f}")
#         print(f"  Z范围: {np.nanmin(z_grid):.2f} ~ {np.nanmax(z_grid):.2f}")

#         return x_coords, y_coords, z_grid

#     except FileNotFoundError:
#         print(f"错误: 找不到文件 {filepath}")
#         return None, None, None
#     except Exception as e:
#         print(f"错误: 读取Surface文件时发生异常: {e}")
#         return None, None, None


# def filter_surface_data_by_z(x_coords, y_coords, z_grid, z_min, z_max, z_buffer_ratio=0.1):
#     """
#     根据Z值范围过滤Surface数据，只保留相关区域的断层面

#     Args:
#         x_coords, y_coords, z_grid: Surface数据
#         z_min, z_max: 解释数据的Z值范围
#         z_buffer_ratio: Z方向缓冲区比例

#     Returns:
#         tuple: 过滤后的断层点坐标 (fault_x, fault_y, fault_z)
#     """
#     if z_grid is None:
#         return np.array([]), np.array([]), np.array([])

#     # 计算Z值缓冲区
#     z_range = z_max - z_min
#     z_buffer = z_range * z_buffer_ratio
#     extended_z_min = z_min - z_buffer
#     extended_z_max = z_max + z_buffer

#     # 创建坐标网格
#     X, Y = np.meshgrid(x_coords, y_coords)

#     # 找到Z值在有效范围内的点
#     valid_mask = (z_grid >= extended_z_min) & (z_grid <= extended_z_max) & (~np.isnan(z_grid))

#     # 提取有效点的坐标
#     fault_x = X[valid_mask]
#     fault_y = Y[valid_mask]
#     fault_z = z_grid[valid_mask]

#     print(f"Surface数据Z值过滤:")
#     print(f"  过滤范围: Z ∈ [{extended_z_min:.2f}, {extended_z_max:.2f}] (含{z_buffer_ratio * 100}%缓冲)")
#     print(f"  原始网格点数: {z_grid.size}")
#     print(f"  有效断层点数: {len(fault_x)}")

#     return fault_x, fault_y, fault_z


def visualize_interpretation_fault(
    interpretation_data_clean, fault_x, fault_y, fault_z, fault_name="未知断层", output_dir=None
):
    """
    可视化断层点和解释数据的分布，并保存图片

    Args:
        interpretation_data_clean: 解释数据
        fault_x, fault_y, fault_z: 断层点坐标
        fault_name: 断层名称
        output_dir: 输出目录
    """

    # 检查有效断层点数量
    if len(fault_x) <= 600:
        print(f"断层有效点数({len(fault_x)})较少，跳过可视化")
        return

    # 显示数据叠加图
    plt.figure(figsize=(10, 8))

    # 解释数据采样显示
    sample_interp = interpretation_data_clean[:: max(1, len(interpretation_data_clean) // 5000)]
    plt.scatter(sample_interp[:, 2], sample_interp[:, 3], s=1, alpha=0.6, c="blue", label="解释数据")

    # 断层点采样显示
    sample_size = min(len(fault_x), 2000)  # 最多显示2000个断层点
    if sample_size < len(fault_x):
        sample_indices = np.random.choice(len(fault_x), sample_size, replace=False)
        fault_sample_x = fault_x[sample_indices]
        fault_sample_y = fault_y[sample_indices]
    else:
        fault_sample_x = fault_x
        fault_sample_y = fault_y

    plt.scatter(fault_sample_x, fault_sample_y, s=3, alpha=0.8, c="red", marker="s", label=f"断层点: {fault_name}")

    # 更新标题，包含断层名称
    plt.title(f"断层数据叠加显示 - {fault_name}")
    plt.xlabel("X坐标")
    plt.ylabel("Y坐标")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 保存图片
    if output_dir:
        # 清理断层名称作为文件名
        safe_fault_name = fault_name.replace(" ", "_").replace("/", "_").replace("\\", "_").replace(".", "_")
        figure_file = os.path.join(output_dir, f"fault_visualization_{safe_fault_name}.png")
        plt.savefig(figure_file, dpi=150, bbox_inches="tight")
        print(f"断层可视化图片已保存: {figure_file}")

    plt.show()


def remove_fault_points(interpretation_data, fault_x, fault_y, fault_z, th1=50.0, th2_ratio=0.05):
    """
    核心处理模块：根据断层点移除解释数据中的断层影响点

    Args:
        interpretation_data (np.ndarray): 解释数据 [inline, xline, x, y, z]
        fault_x, fault_y, fault_z (np.ndarray): 断层点坐标
        th1 (float): 水平距离阈值（米）
        th2_ratio (float): 垂直距离阈值比例

    Returns:
        tuple: (保留的数据, 被移除的数据)
    """
    if len(interpretation_data) == 0 or len(fault_x) == 0:
        print("警告: 输入数据为空，跳过断层擦除")
        return interpretation_data, np.array([])

    # 计算垂直距离阈值
    z_range = interpretation_data[:, 4].max() - interpretation_data[:, 4].min()
    th2 = z_range * th2_ratio

    print(f"断层擦除参数:")
    print(f"  水平距离阈值 (th1): {th1:.1f} 米")
    print(f"  垂直距离阈值 (th2): {th2:.1f} ms ({th2_ratio * 100:.1f}% × Z范围)")
    print(f"  解释点数量: {len(interpretation_data)}")
    print(f"  断层点数量: {len(fault_x)}")

    # 提取解释点的坐标
    interp_x = interpretation_data[:, 2]  # X坐标
    interp_y = interpretation_data[:, 3]  # Y坐标
    interp_z = interpretation_data[:, 4]  # Z坐标

    # 标记要保留的点
    keep_mask = np.ones(len(interpretation_data), dtype=bool)

    # 双层循环：遍历每个解释点
    for i in range(len(interpretation_data)):
        # 当前解释点的坐标
        curr_x, curr_y, curr_z = interp_x[i], interp_y[i], interp_z[i]

        # 计算与所有断层点的距离
        # 水平距离（XY平面）
        horizontal_dist = np.sqrt((fault_x - curr_x) ** 2 + (fault_y - curr_y) ** 2)

        # 垂直距离（Z方向）
        vertical_dist = np.abs(fault_z - curr_z)

        # 检查是否有断层点在阈值范围内
        close_points = (horizontal_dist <= th1) & (vertical_dist <= th2)

        # 如果有断层点在影响范围内，标记该解释点为删除
        if np.any(close_points):
            keep_mask[i] = False

    # 分离保留和移除的数据
    kept_data = interpretation_data[keep_mask]
    removed_data = interpretation_data[~keep_mask]

    print(f"断层擦除完成:")
    print(f"  原始解释点: {len(interpretation_data)} 个")
    print(f"  保留点数: {len(kept_data)} 个 ({len(kept_data) / len(interpretation_data) * 100:.1f}%)")
    print(f"  移除点数: {len(removed_data)} 个 ({len(removed_data) / len(interpretation_data) * 100:.1f}%)")

    return kept_data, removed_data


def process_single_fault(
    interpretation_data_clean,
    fault_file,
    fault_index,
    total_faults,
    th1=50.0,
    th2_ratio=0.05,
    z_buffer_ratio=0.1,
    visualize=True,
    output_dir=None,
):
    """
    处理单个断层的擦除流程

    Args:
        interpretation_data_clean: 清理后的解释数据
        fault_file: 断层文件路径
        fault_index: 当前断层索引
        total_faults: 总断层数
        th1: 水平距离阈值
        th2_ratio: 垂直距离阈值比例
        z_buffer_ratio: Z值缓冲区比例
        visualize: 是否生成可视化
        output_dir: 输出目录

    Returns:
        tuple: (处理后的数据, 移除的数据, 断层名称)
    """
    fault_name = os.path.basename(fault_file)
    print(f"\n{'=' * 80}")
    print(f"处理断层 {fault_index}/{total_faults}: {fault_name}")
    print(f"{'=' * 80}")

    # 加载断层数据
    print(f"加载断层点数据...")
    fault_x, fault_y, fault_z = load_fault_points(fault_file)

    if len(fault_x) == 0:
        print(f"错误: 无法加载断层文件 {fault_name}")
        return interpretation_data_clean, np.array([]), fault_name

    # Z值过滤
    z_min = interpretation_data_clean[:, 4].min()
    z_max = interpretation_data_clean[:, 4].max()

    print(f"解释数据Z值范围: [{z_min:.2f}, {z_max:.2f}]")

    filtered_fault_x, filtered_fault_y, filtered_fault_z = filter_fault_points_by_z(
        fault_x, fault_y, fault_z, z_min, z_max, z_buffer_ratio=z_buffer_ratio
    )

    if len(filtered_fault_x) == 0:
        print(f"警告: 断层 {fault_name} 在有效Z值范围内无数据，跳过")
        return interpretation_data_clean, np.array([]), fault_name

    # 可视化验证（仅针对前几个断层，避免过多图形）
    if visualize:
        print(f"生成断层 {fault_name} 的可视化...")
        visualize_interpretation_fault(
            interpretation_data_clean, filtered_fault_x, filtered_fault_y, filtered_fault_z, fault_name, output_dir
        )

    # 执行断层擦除
    print(f"执行断层擦除算法...")
    kept_data, removed_data = remove_fault_points(
        interpretation_data_clean, filtered_fault_x, filtered_fault_y, filtered_fault_z, th1=th1, th2_ratio=th2_ratio
    )

    print(f"断层 {fault_name} 处理完成:")
    print(f"  输入点数: {len(interpretation_data_clean):,}")
    print(f"  保留点数: {len(kept_data):,}")
    print(f"  移除点数: {len(removed_data):,}")
    print(f"  移除率: {len(removed_data) / len(interpretation_data_clean) * 100:.1f}%")

    return kept_data, removed_data, fault_name

In [ ]:
def save_results(filepath, interpretation_data):
    """
    保存处理后的解释数据

    Args:
        filepath (str): 输出文件路径
        interpretation_data (np.ndarray): 处理后的解释数据
    """
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            for row in interpretation_data:
                inline, xline, x, y, z = row
                # 格式化输出，保持与原始格式一致
                line = f"INLINE :{inline:7.0f} XLINE :{xline:7.0f}{x:15.5f}{y:15.5f}{z:15.5f}\n"
                f.write(line)

        print(f"成功保存结果到: {filepath}")
        print(f"保存数据点数: {len(interpretation_data)}")

    except Exception as e:
        print(f"错误: 保存文件时发生异常: {e}")


# def save_intermediate_results(output_dir, surface_name, fault_name, kept_data, removed_data, fault_index):
#     """
#     保存中间处理结果
#     """
#     # 清理断层名称作为文件名
#     safe_fault_name = fault_name.replace(" ", "_").replace("/", "_").replace("\\", "_")

#     # 保存当前步骤的结果
#     kept_file = os.path.join(output_dir, f"{surface_name}_after_{fault_index:02d}_{safe_fault_name}_kept.txt")
#     removed_file = os.path.join(output_dir, f"{surface_name}_after_{fault_index:02d}_{safe_fault_name}_removed.txt")

#     if len(kept_data) > 0:
#         save_results(kept_file, kept_data)

#     if len(removed_data) > 0:
#         save_results(removed_file, removed_data)

#     return kept_file, removed_file


def visualize_xy_distribution(interpretation_data_clean, kept_data, removed_data, fault_x, fault_y, sample_ratio=0.1):
    """
    可视化XY平面分布

    Args:
        interpretation_data_clean: 原始清理后的解释数据
        kept_data: 保留的解释数据
        removed_data: 被移除的解释数据
        fault_x, fault_y: 断层点XY坐标
        sample_ratio: 采样比例（用于减少显示点数）
    """

    # 数据采样（避免图形过于密集）
    def sample_data(data, ratio):
        if len(data) == 0:
            return data
        n_samples = max(1, int(len(data) * ratio))
        indices = np.random.choice(len(data), n_samples, replace=False)
        return data[indices]

    # 采样数据
    kept_sample = sample_data(kept_data, sample_ratio)
    removed_sample = sample_data(removed_data, sample_ratio)
    fault_sample_indices = np.random.choice(len(fault_x), max(1, int(len(fault_x) * sample_ratio)), replace=False)

    plt.figure(figsize=(12, 8))

    # XY平面分布
    plt.scatter(kept_sample[:, 2], kept_sample[:, 3], s=0.5, alpha=0.6, c="blue", label=f"保留点 ({len(kept_data):,})")
    plt.scatter(
        removed_sample[:, 2], removed_sample[:, 3], s=0.8, alpha=0.8, c="red", label=f"移除点 ({len(removed_data):,})"
    )
    plt.scatter(
        fault_x[fault_sample_indices],
        fault_y[fault_sample_indices],
        s=3,
        alpha=0.7,
        c="orange",
        marker="s",
        label=f"断层点 ({len(fault_x):,})",
    )

    plt.xlabel("X坐标 (m)")
    plt.ylabel("Y坐标 (m)")
    plt.title("断层擦除结果 - XY平面分布")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 添加统计信息
    total_original = len(interpretation_data_clean)
    total_kept = len(kept_data)
    total_removed = len(removed_data)
    removal_percentage = total_removed / total_original * 100

    stats_text = f"""统计信息:
原始: {total_original:,} 点
保留: {total_kept:,} 点 ({(total_kept / total_original) * 100:.1f}%)
移除: {total_removed:,} 点 ({removal_percentage:.1f}%)
断层: {len(fault_x):,} 点"""

    plt.text(
        0.02,
        0.98,
        stats_text,
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )

    plt.tight_layout()
    plt.show()

    # 打印详细统计
    print("\n" + "=" * 60)
    print("断层擦除结果统计")
    print("=" * 60)
    print(f"原始解释点数量: {total_original:,}")
    print(f"断层影响区域移除: {total_removed:,} 点 ({removal_percentage:.1f}%)")
    print(f"保留清洁数据: {total_kept:,} 点 ({(total_kept / total_original) * 100:.1f}%)")
    print(f"断层点参考数量: {len(fault_x):,}")
    print("=" * 60)


In [ ]:
# # 测试用代码
# print("=" * 60)
# print("地震解释自动加密 - 断层擦除工具 (左上角模式)")
# print("=" * 60)

# # 文件路径配置
# interpretation_file = f"{data_dir}/{SURFACE_NAME}"
# surface_file = f"{data_dir}/F1_surface"
# output_file = f"{output_dir}/{SURFACE_NAME}_cleaned"

# print(f"\n当前处理层位: {SURFACE_NAME}")
# print(f"解释文件: {interpretation_file}")
# print(f"断层文件: {surface_file}")

# # 1. 加载解释数据
# print("\n步骤1: 加载解释数据...")
# interpretation_data = load_interpretation_data(interpretation_file)

# if len(interpretation_data) == 0:
#     print("错误: 无法加载解释数据")
# else:
#     # 2. 离群值处理
#     print("\n步骤2: 离群值检测和清理...")
#     interpretation_data_clean = remove_outliers_by_z(interpretation_data, method="iqr", factor=1.5)

#     if len(interpretation_data_clean) == 0:
#         print("错误: 离群值处理后无数据残留")
#     else:
#         # 3. 加载断层数据（左上角模式）
#         print("\n步骤3: 加载断层Surface数据（左上角模式）...")
#         x_coords, y_coords, z_grid = load_surface_data_top_left(surface_file)

#         if z_grid is not None:
#             # 4. 可视化验证
#             print("\n步骤4: 可视化验证坐标系统...")
#             visualize_fault_orientation(interpretation_data_clean, x_coords, y_coords, z_grid)

#             # 5. 数据预处理 - Z值过滤
#             print("\n步骤5: 断层数据预处理...")
#             z_min = interpretation_data_clean[:, 4].min()
#             z_max = interpretation_data_clean[:, 4].max()

#             print(f"清理后解释数据Z值范围: [{z_min:.2f}, {z_max:.2f}]")

#             fault_x, fault_y, fault_z = filter_surface_data_by_z(
#                 x_coords, y_coords, z_grid, z_min, z_max, z_buffer_ratio=0.1
#             )

#             if len(fault_x) > 0:
#                 # 6. 执行断层擦除算法
#                 print("\n步骤6: 执行断层擦除算法...")

#                 # 设置参数
#                 th1 = 50.0  # 水平距离阈值（米）
#                 th2_ratio = 0.05  # 垂直距离阈值比例（5%的Z值范围）

#                 # 执行断层擦除
#                 kept_data, removed_data = remove_fault_points(
#                     interpretation_data_clean, fault_x, fault_y, fault_z, th1=th1, th2_ratio=th2_ratio
#                 )

#                 # 7. 保存最终结果
#                 if len(kept_data) > 0:
#                     print(f"\n步骤7: 保存最终结果...")
#                     save_results(output_file, kept_data)

#                     # 8. 可视化结果
#                     print(f"\n步骤8: 生成可视化结果...")
#                     visualize_xy_distribution(
#                         interpretation_data_clean,
#                         kept_data,
#                         removed_data,
#                         fault_x,
#                         fault_y,
#                         sample_ratio=0.05,
#                     )

#                     print(f"\n处理完成! 最终文件已保存至: {output_file}")

#                     # 最终汇总
#                     print("\n" + "=" * 60)
#                     print("完整处理流程汇总")
#                     print("=" * 60)
#                     print(f"原始解释数据: {len(interpretation_data):,} 点")
#                     print(f"离群值清理后: {len(interpretation_data_clean):,} 点")
#                     print(f"断层擦除后: {len(kept_data):,} 点")
#                     print(f"总移除率: {(1 - len(kept_data) / len(interpretation_data)) * 100:.1f}%")
#                     print(f"最终输出文件: {output_file}")
#                     print("=" * 60)
#                 else:
#                     print("错误: 断层擦除后无数据残留!")
#             else:
#                 print("警告: 没有有效的断层点数据，跳过断层擦除步骤")
#         else:
#             print("错误: 无法加载断层Surface数据")

In [ ]:
# ==================== 批量断层擦除流程 ====================
print("=" * 80)
print("地震解释自动加密 - 批量断层擦除工具")
print("=" * 80)

# 1. 加载并预处理解释数据
print("\n步骤1: 加载解释数据...")
interpretation_file = os.path.join(interpre_dir, SURFACE_NAME)
interpretation_data = load_interpretation_data(interpretation_file)

if len(interpretation_data) == 0:
    print("错误: 无法加载解释数据，程序退出")
else:
    print("\n步骤2: 离群值检测和清理...")
    interpretation_data_clean = remove_interpretation_outliers_by_z(interpretation_data, method="iqr", factor=3.0)

    if len(interpretation_data_clean) == 0:
        print("错误: 离群值处理后无数据残留，程序退出")
    else:
        # 2. 获取所有断层文件
        print("\n步骤3: 扫描断层文件...")
        fault_files = get_fault_files(fault_dir)

        if len(fault_files) == 0:
            print("警告: 未找到断层文件，跳过断层擦除步骤")
            final_data = interpretation_data_clean
        else:
            # 3. 批量处理断层（累积擦除）
            print(f"\n步骤4: 开始批量处理 {len(fault_files)} 个断层...")

            # 初始化处理数据
            current_data = interpretation_data_clean.copy()
            total_removed_data = []
            processing_summary = []

            # 设置处理参数
            th1 = 50.0  # 水平距离阈值（米）
            th2_ratio = 0.05  # 垂直距离阈值比例
            z_buffer_ratio = 0.1  # Z值缓冲区比例

            # 逐个处理断层
            for fault_index, fault_file in enumerate(fault_files, 1):
                print(f"\n{'=' * 60}")
                print(f"处理断层 {fault_index}/{len(fault_files)}")
                print(f"{'=' * 60}")

                # 处理单个断层
                kept_data, removed_data, fault_name = process_single_fault(
                    current_data,
                    fault_file,
                    fault_index,
                    len(fault_files),
                    th1=th1,
                    th2_ratio=th2_ratio,
                    z_buffer_ratio=z_buffer_ratio,
                    visualize=True,
                    output_dir=output_dir,  # 传递输出目录用于保存图片
                )

                # 记录处理摘要
                processing_summary.append(
                    {
                        "fault_index": fault_index,
                        "fault_name": fault_name,
                        "input_points": len(current_data),
                        "kept_points": len(kept_data),
                        "removed_points": len(removed_data),
                        "removal_rate": len(removed_data) / len(current_data) * 100 if len(current_data) > 0 else 0,
                    }
                )

                # 累积移除的数据
                if len(removed_data) > 0:
                    total_removed_data.extend(removed_data)

                # 更新当前数据为下一轮处理的输入
                current_data = kept_data.copy()

                # 如果数据被完全移除，提前结束
                if len(current_data) == 0:
                    print(f"警告: 处理断层 {fault_name} 后无数据残留，停止后续处理")
                    break

            final_data = current_data

            # 4. 保存最终结果
            print(f"\n步骤5: 保存最终处理结果...")
            if len(final_data) > 0:
                final_output_file = os.path.join(output_dir, f"{SURFACE_NAME}_final_cleaned.txt")
                save_results(final_output_file, final_data)

                # 保存所有被移除的数据
                if total_removed_data:
                    removed_output_file = os.path.join(output_dir, f"{SURFACE_NAME}_all_removed.txt")
                    save_results(removed_output_file, np.array(total_removed_data))

                # 5. 生成处理摘要报告
                print(f"\n步骤6: 生成处理摘要报告...")
                summary_df = pd.DataFrame(processing_summary)
                summary_file = os.path.join(output_dir, f"{SURFACE_NAME}_processing_summary.csv")
                summary_df.to_csv(summary_file, index=False)
                print(f"处理摘要已保存至: {summary_file}")

                # 6. 最终可视化（如果有断层文件的话）
                if len(fault_files) > 0:
                    print(f"\n步骤7: 生成最终可视化结果...")
                    # 重新加载最后一个断层数据用于可视化
                    last_fault_file = fault_files[-1]

                    fault_x, fault_y, fault_z = load_fault_points(last_fault_file)

                    if len(fault_x) > 0:
                        z_min = final_data[:, 4].min()
                        z_max = final_data[:, 4].max()

                        filtered_fault_x, filtered_fault_y, filtered_fault_z = filter_fault_points_by_z(
                            fault_x, fault_y, fault_z, z_min, z_max, z_buffer_ratio=z_buffer_ratio
                        )

                        # 可视化最终结果
                        visualize_xy_distribution(
                            interpretation_data_clean,
                            final_data,
                            np.array(total_removed_data) if total_removed_data else np.array([]),
                            filtered_fault_x,
                            filtered_fault_y,
                            sample_ratio=0.05,
                        )
                    else:
                        print("警告: 无法加载最后一个断层文件用于可视化")

                # 7. 打印最终统计
                print("\n" + "=" * 80)
                print("批量断层擦除完整统计")
                print("=" * 80)
                print(f"处理层位: {SURFACE_NAME}")
                print(f"处理断层数量: {len(fault_files)}")
                print(f"原始解释数据: {len(interpretation_data):,} 点")
                print(f"离群值清理后: {len(interpretation_data_clean):,} 点")
                print(f"最终保留数据: {len(final_data):,} 点")
                print(f"总移除数据: {len(total_removed_data):,} 点")
                print(f"总移除率: {(len(total_removed_data) / len(interpretation_data_clean)) * 100:.1f}%")
                print(f"数据保留率: {(len(final_data) / len(interpretation_data_clean)) * 100:.1f}%")
                print(f"\n输出文件:")
                print(f"  - 最终清理数据: {final_output_file}")
                if total_removed_data:
                    print(f"  - 所有移除数据: {removed_output_file}")
                print(f"  - 处理摘要: {summary_file}")
                print(f"  - 输出目录: {output_dir}")

                # 打印每个断层的处理摘要
                print(f"\n各断层处理详情:")
                for summary in processing_summary:
                    print(
                        f"  {summary['fault_index']:2d}. {summary['fault_name']}: "
                        f"{summary['removed_points']:,} 移除 "
                        f"({summary['removal_rate']:.1f}%), "
                        f"{summary['kept_points']:,} 保留"
                    )

                print("=" * 80)
                print("批量处理完成!")
                print("=" * 80)

            else:
                print("警告: 所有断层处理完成后无数据残留!")